# Fault and downtime analysis (diagnostic)

Supporting analysis for PTH Tier 1 metrics **ROSP-11** (time-loss accounting) and
**ROSP-12** (fault rate, MTTR and top fault categories) - {{ params.start_dayobs }} to
{{ params.end_dayobs }} (dayObs, inclusive).

This is a **diagnostic** notebook, not an official metric page. It develops the per-night
time attribution and fault statistics that both metrics depend on, as input to the time
attribution proposal in [ROSP-24](https://rubinobs.atlassian.net/browse/ROSP-24). The
official pages remain `rosp-11_time_loss_accounting` and `rosp-12_fault_rate_mttr`.

- Jira epics: [ROSP-11](https://rubinobs.atlassian.net/browse/ROSP-11),
  [ROSP-12](https://rubinobs.atlassian.net/browse/ROSP-12)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

Steward: Brian Stalder. Method: in development, not baselined; nothing here is a frozen
metric value.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
# Defaults are the PTH baseline epoch.
start_dayobs = 20260629  # First dayObs (inclusive)
end_dayobs = 20261231  # Last dayObs (inclusive)

# Note: the EFD observatoryStatus topic this notebook is built on only begins
# 20260225. Earlier nights are backfilled from the narrative log for the hours
# budget, and are excluded from the per-component fault statistics.

not_times_square = True

In [ ]:
# Times Square runs this notebook in a Nublado pod where rubin_nights
# may be missing or stale; install/upgrade it there. When running
# anywhere else (e.g. an SDF login node) this cell is a no-op.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print("updating rubin_nights")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

# Import here rather than relying on the parameters cell above, which Times Square
# replaces wholesale with sidecar values (dropping any imports it contained).
import rubin_nights.dayobs_utils as rn_dayobs

# Times Square passes parameters as strings. The rest of the notebook wants ints:
# f-string SQL and str(day_obs) slicing tolerate strings, but rn_dayobs and the
# %i format specifiers below do not. day_obs_str_to_int accepts both "20260629"
# and the dashed "2026-06-29" that a `format: dayobs` parameter would arrive as.
start_dayobs = rn_dayobs.day_obs_str_to_int(str(start_dayobs))
end_dayobs = rn_dayobs.day_obs_str_to_int(str(end_dayobs))

# Analyse complete nights only. Nights after yesterday's dayObs have not been
# observed yet; left in, they add empty bars and are reported as "covered".
requested_end_dayobs = end_dayobs
end_dayobs = min(
    end_dayobs, rn_dayobs.day_obs_str_to_int(rn_dayobs.yesterday_day_obs())
)
if end_dayobs < requested_end_dayobs:
    print(
        f"Requested range ends {requested_end_dayobs}; analysing through "
        f"{end_dayobs}, the last complete night."
    )
if start_dayobs > end_dayobs:
    raise ValueError(f"start_dayobs {start_dayobs} is after {end_dayobs}")

In [ ]:
SAVE_FIGURES = False
# Survey programs, drawn as ticks on the state timeline so a gap can be read
# off against what the observatory was doing.
SURVEY_PROGRAMS = [
    "BLOCK-407",
    "BLOCK-408",
    "BLOCK-416",
    "BLOCK-417",
    "BLOCK-419",
    "BLOCK-421",
]  # Early Ops

# Fault and Downtime Analysis

Where the lost on-sky time went. `on-sky_utilization.ipynb` measures how well the time
*between consecutive visits* was used, so it is blind to the nights that cost the most: over
a long range several nights have no LSSTCam visits at all, and therefore no gaps. This
notebook accounts for the whole night instead. For every `day_obs` in the
`start_dayobs`–`end_dayobs` range it:

1. **Partitions the night** into operational time versus time lost to weather, downtime, a
   fault, or idling, from the EFD `lsst.sal.Scheduler.logevent_observatoryStatus` topic, so
   that the per-night hours sum exactly to the length of the $-12^\circ$ night.
2. **Attributes fault hours to a subsystem**, by parsing the faulting CSC names out of the
   scheduler's status notes.
3. **Turns those faults into reliability statistics** — fault rate per hour of exposure,
   mean time between faults, and recovery time — per component, and trends them over the
   range.
4. **Cross-checks** the partition against the overlapping `rubin_nights` observatory states
   and the hand-written narrative log, and reconciles the differences.

`start_dayobs` / `end_dayobs` are supplied by the sidecar YAML when run as a Times Square
page; the parameters cell above sets local defaults and the `not_times_square` runtime flag.

Note that the EFD topic this notebook is built on begins **2026-02-25**. Over a range reaching
earlier than that, those nights are backfilled from the narrative log for the hours budget and
excluded from the per-component statistics; the notebook detects the boundary and reports it
rather than assuming a fixed date. The same applies to **gaps** in the topic, such as the one
that opened after the July 2026 storm: a state is trusted for at most 30 hours after its
message, and longer silences are treated as unmeasured rather than carried forward. Periods
that only people can label, like storm recovery or scheduled downtime, come from a short
table of curated annotations.

In [ ]:
import re

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from astropy.time import Time, TimeDelta
import astropy.units as u

from IPython.display import display

import rubin_nights.dayobs_utils as rn_dayobs
import rubin_nights.observatory_status as observatory_status

In [ ]:
from lsst.utils.plotting import publication_plots

publication_plots.set_rubin_plotstyle()

In [ ]:
# Connect via rubin_nights, which resolves the correct endpoints and RSP token
# for whatever environment this runs in (Times Square / Nublado in-cluster, the
# USDF RSP, or an SDF login node). This notebook needs three of them: the
# ConsDB for visits, the EFD for observatory status, and the narrative log.
from rubin_nights import connections

clients = connections.get_clients()  # site auto-detected, token auto-resolved
client = clients["consdb"]
efd_client = clients["efd"]

In [ ]:
# Visits are used only to place survey ticks on the state timeline and to find
# inter-visit gaps for the event table, so pull just the columns those need.
instrument = "lsstcam"


def fetch(start_dayobs, end_dayobs):
    visits_query = f"""
    SELECT
        day_obs, science_program, obs_start_mjd, obs_end_mjd, exp_midpt_mjd
    FROM
        cdb_{instrument}.visit1 AS visit1
    WHERE
        airmass > 0.
        AND img_type in ('acq', 'science', 'cwfs', 'focus')
        AND day_obs >= {start_dayobs} AND day_obs <= {end_dayobs}
    """
    return client.query(visits_query).sort_values(by="obs_start_mjd")


visits = fetch(start_dayobs, end_dayobs)
print(f"{len(visits)} visits on {visits.day_obs.nunique()} nights")

In [ ]:
# Time conventions shared with on-sky_utilization.ipynb, so the state timeline
# below lines up with the visit timeline there: the x-axis is the MJD of the
# day_obs at 0h UTC, and the y-axis is hours after the *following* 0h UTC.


def dayObsToTime(day_obs):
    year = str(day_obs)[0:4]
    month = str(day_obs)[4:6]
    day = str(day_obs)[6:8]
    time = Time("%s-%s-%s 00:00:00" % (year, month, day), format="iso")
    return time


def convertTime(time_input, time_reference):
    time_output = (time_input - (time_reference + (1.0 * u.day))) * u.day.to(u.hr)
    return time_output.value

## Building the lost-time partition

The source is the EFD topic `lsst.sal.Scheduler.logevent_observatoryStatus`, fetched via
`rubin_nights.observatory_status.get_obs_status_messages`. Each message carries
`statusLabels` (a pipe-joined set of `OPERATIONAL` / `IDLE` / `FAULT` / `WEATHER` /
`DOWNTIME` / `UNKNOWN`) and a `note` whose machine-generated clause names the faulting CSCs
("The following components are in `<State.FAULT: 3>` state: MTMount, MTPtg").

**Building an exclusive partition.** For each `day_obs` the state in force at $-12^\circ$
sunset is seeded from the last message at or before it; each subsequent message's state runs
until the next message or $-12^\circ$ sunrise. Because several labels can be set at once,
each interval is reduced to a single `cause` by the priority

`DOWNTIME > WEATHER > FAULT > IDLE > OPERATIONAL > UNKNOWN`

so the per-night hours sum to `night_hours` exactly (asserted below). This deliberately
differs from `observatory_status.get_observatory_state_times`, whose periods **overlap** and
cannot be summed — a `WEATHER` period typically spans the whole night while
`OPERATIONAL` / `IDLE` / `FAULT` run inside it. That function is still used at the end of the
notebook as an independent cross-check.

Weather and downtime outrank fault, so a fault occurring while the dome was already shut for
weather is charged to weather; those hours are reported separately rather than double
counted. Two further conventions to keep in mind when reading the numbers:

- `FAULT` hours include **intentional engineering faults** — for example, one note reads
  "We were intentionally faulting the mirror to perform some tests. Not an actual fault."
  on a message still labelled `FAULT`. Rather than filter on operator prose, the raw note is
  surfaced in the event table below.
- When a fault run names several CSCs, its hours are **split equally** among them. That is a
  convention, not a measurement; `n_cscs` records where it was applied.
- `UNKNOWN` is kept as its own bucket rather than being bridged into the surrounding state
  (which is what `get_observatory_state_times` does), since a partition cannot relabel it.

**Telemetry gaps.** The Scheduler emits a message at least once a day while it is running,
so a state is trusted for at most `MAX_STATE_AGE_HOURS` (30 h) after its message. Beyond
that the time is recorded as unmeasured (`provenance = no_telemetry`) instead of inheriting
a state that may be weeks old. Without this, the last message before a long outage is
copied across every night of it. A night counts as **EFD-covered** only when at least half
of it has fresh telemetry, and only covered nights feed the fault statistics.

**Curated annotations.** `TIME_LOSS_ANNOTATIONS` records periods the automated sources
cannot label, each with its source. An annotation fills unmeasured time in its range and
relabels time the EFD reports as `DOWNTIME`, `WEATHER` or `UNKNOWN`. It never overwrites
recorded `OPERATIONAL`, `FAULT` or `IDLE` time, so an open-ended annotation cannot hide
observing once it resumes. Storm recovery is kept as its own `RECOVERY` cause until the
time-attribution proposal ([ROSP-24](https://rubinobs.atlassian.net/browse/ROSP-24))
decides where it belongs.

In [ ]:
# Observatory-state attribution: constants and helpers.

# Reduction order for a message with several labels set at once. Highest first:
# weather/downtime outrank a fault, so a fault while the dome was already shut
# for weather is charged to weather (and reported separately, not twice).
CAUSE_PRIORITY = ("DOWNTIME", "WEATHER", "FAULT", "IDLE", "OPERATIONAL", "UNKNOWN")

# Causes that only curated annotations assign; the EFD never emits them.
ANNOTATION_CAUSES = ("RECOVERY",)
# Column and plotting order for the hours budget.
BUDGET_CAUSES = (
    "DOWNTIME",
    "RECOVERY",
    "WEATHER",
    "FAULT",
    "IDLE",
    "OPERATIONAL",
    "UNKNOWN",
)
LOST_CAUSES = ("DOWNTIME", "RECOVERY", "WEATHER", "FAULT", "IDLE")

# A status message's state is trusted for at most this long. The Scheduler emits
# at least one message a day while running, so a longer silence means it was not
# reporting, and carrying the old state forward would invent hours.
MAX_STATE_AGE_HOURS = 30.0
# A night is EFD-covered when at least this fraction of it has fresh telemetry.
MIN_COVERED_FRACTION = 0.5

# EFD causes an annotation may relabel. Recorded OPERATIONAL / FAULT / IDLE time
# is never overwritten.
ANNOTATION_OVERRIDES = ("DOWNTIME", "WEATHER", "UNKNOWN")

# Curated periods the automated sources cannot label. Inclusive dayObs; an `end`
# of None runs to the end of the analysed range.
TIME_LOSS_ANNOTATIONS = [
    {
        "start": 20260715,
        "end": 20260913,
        "cause": "RECOVERY",
        "label": "storm recovery",
        "source": "B. Stalder 2026-10-11: recovery from the July 2026 winter storm, "
        "until scheduled downtime began 2026-09-14. EFD observatoryStatus is silent "
        "2026-07-20 to 2026-09-25.",
    },
    {
        "start": 20260914,
        "end": None,
        "cause": "DOWNTIME",
        "label": "scheduled downtime",
        "source": "B. Stalder 2026-10-11: scheduled downtime began 2026-09-14.",
    },
]

CAUSE_COLORS = {
    "OPERATIONAL": "tab:blue",
    "FAULT": "tab:red",
    "WEATHER": "tab:cyan",
    "DOWNTIME": "tab:purple",
    "IDLE": "tab:orange",
    "UNKNOWN": "0.6",
    "RECOVERY": "tab:pink",
}

# Faulting CSC -> engineering subsystem, keyed on the *base* name so that
# SAL-indexed CSCs (MTHexapod:1, Scheduler:1) resolve via name.split(":")[0].
CSC_SUBSYSTEMS = {
    "MTMount": "TMA",
    "MTPtg": "TMA",
    "MTRotator": "TMA",
    "MTM1M3": "M1M3",
    "MTM2": "M2/hexapods",
    "MTHexapod": "M2/hexapods",
    "MTAOS": "AOS",
    "MTCamera": "Camera",
    "MTOODS": "Camera",
    "MTHeaderService": "Camera",
    "MTDome": "Dome",
    "Scheduler": "Scheduler/queue",
    "ScriptQueue": "Scheduler/queue",
}
UNATTRIBUTED = "unattributed"

SUBSYSTEM_COLORS = {
    "TMA": "tab:red",
    "M1M3": "tab:orange",
    "M2/hexapods": "gold",
    "AOS": "tab:green",
    "Camera": "tab:blue",
    "Dome": "tab:purple",
    "Scheduler/queue": "tab:brown",
    UNATTRIBUTED: "0.6",
}

# Match only the machine-generated clause, and only its *current* ("are in")
# form - "were in" is sticky and keeps repeating long after recovery. Operator
# free text may precede the clause and must not be searched.
FAULT_CSC_RE = re.compile(r"components are in <State\.FAULT: \d+> state:\s*([^.]*)\.")


def fetch_status_messages(start_dayobs, end_dayobs, efd_client, lead_days=3):
    """Observatory status messages, with a lead-in so the state in force at the
    first sunset is known even when the last message predates the range."""
    messages = observatory_status.get_obs_status_messages(
        rn_dayobs.day_obs_to_time(start_dayobs) - TimeDelta(lead_days, format="jd"),
        rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
        efd_client,
    )
    # searchsorted below returns garbage, silently, on an unsorted index.
    return messages.sort_index()


def exclusive_cause(status_labels):
    """The single highest-priority cause in a pipe-joined label string.

    Label order varies in the EFD ("FAULT | WEATHER" and "WEATHER | IDLE" both
    occur), so this tests for substrings rather than comparing whole strings.
    """
    for cause in CAUSE_PRIORITY:
        if cause in status_labels:
            return cause
    return "UNKNOWN"


def parse_fault_cscs(note):
    """Faulting CSC names (SAL index retained) from a status-message note.

    Two payload formats occur across 2026: the Scheduler emitted a Python-repr
    list ("state: ['MTMount', 'MTPtg'].") up to 2026-06-08 and a bare
    comma-separated list ("state: MTMount, MTPtg.") after. Strip the list
    punctuation so both yield the same names - otherwise the brackets and quotes
    become part of the name and one component splits into several phantom ones.
    """
    if not isinstance(note, str):
        return []
    names = set()
    for match in FAULT_CSC_RE.finditer(note):
        payload = match.group(1).strip().strip("[]")
        # Split on "," only - never on ":", which carries the SAL index.
        for name in payload.split(","):
            name = name.strip().strip("'\"").strip()
            if name:
                names.add(name)
    # A single clause can repeat a CSC ("Scheduler:1, MTAOS, Scheduler:1, ...").
    return sorted(names)


def csc_subsystem(csc):
    return CSC_SUBSYSTEMS.get(csc.split(":")[0], UNATTRIBUTED)

In [ ]:
# Everything below needs the EFD. get_clients() falls back rather than
# returning None when the EFD is unreachable, so guard on an empty result and
# let the notebook finish rather than failing the whole page.
day_obs_nights = rn_dayobs.day_obs_list(
    rn_dayobs.day_obs_to_time(start_dayobs),
    # day_obs_list is end-exclusive, so reach past end_dayobs to include it.
    rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
)

status_messages = fetch_status_messages(start_dayobs, end_dayobs, efd_client)
have_status = len(status_messages) > 0

print(
    f"{len(day_obs_nights)} nights ({start_dayobs}-{end_dayobs}), "
    f"{len(status_messages)} observatory status messages"
)
if not have_status:
    print("No observatory status messages found - skipping the analysis below.")

In [ ]:
def build_night_windows(day_obs_list, sun_alt=-12):
    """-12 deg sunset/sunrise per day_obs, as TAI MJD floats.

    `rn_dayobs.day_obs_sunset_sunrise` is `lru_cache(maxsize=100)`, so a range
    longer than 100 nights thrashes the cache and re-runs astroplan on every
    call - ~9 s per full pass, several passes per notebook. Resolve each night
    once here and pass the result down.
    """
    windows = {}
    for day_obs in day_obs_list:
        sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(day_obs, sun_alt)
        windows[day_obs] = (sunset.tai.mjd, sunrise.tai.mjd)
    return windows


def build_state_intervals(
    messages,
    day_obs_list,
    night_windows,
    clamp_now=True,
    max_age_hours=MAX_STATE_AGE_HOURS,
):
    """Partition each night into non-overlapping observatory-state intervals.

    Returns one row per state run per night, clipped to -12 deg sunset/sunrise,
    with exactly one `cause` each - so hours per night sum to night_hours.
    A message's state is trusted for at most `max_age_hours`; time past that,
    or before the first message, is UNKNOWN with provenance "no_telemetry".
    """
    # Work in TAI MJD floats throughout: the EFD index is tz-aware pandas,
    # which does not compare cleanly against astropy Time.
    msg_mjd = Time(messages.index.to_pydatetime(), scale="utc").tai.mjd
    labels = messages.statusLabels.astype(str).values
    notes = messages.note.values
    now_mjd = Time.now().tai.mjd
    max_age = max_age_hours / 24.0

    rows = []

    def add(day_obs, start, end, cause, label, note, provenance):
        if end > start:
            rows.append(
                {
                    "day_obs": day_obs,
                    "start_mjd": start,
                    "end_mjd": end,
                    "hours": (end - start) * 24,
                    "cause": cause,
                    "status_labels": label,
                    "note": note,
                    "provenance": provenance,
                }
            )

    for day_obs in day_obs_list:
        lo, hi = night_windows[day_obs]
        if clamp_now:
            if lo >= now_mjd:
                continue  # night has not started yet
            hi = min(hi, now_mjd)  # night in progress: count only elapsed time

        # Seed from the last message at or before sunset, then take every
        # message falling inside the night.
        first = max(np.searchsorted(msg_mjd, lo, side="right") - 1, 0)
        last = np.searchsorted(msg_mjd, hi, side="left")
        idx = list(range(first, last))

        # No state known before the first usable message. Record it rather than
        # emitting nothing, which would let the night vanish from the budget
        # while the closure check trivially passed.
        edge = hi if len(idx) == 0 else max(min(msg_mjd[idx[0]], hi), lo)
        add(day_obs, lo, edge, "UNKNOWN", "", None, "no_telemetry")

        for i in idx:
            start = max(msg_mjd[i], lo)
            end = min(msg_mjd[i + 1], hi) if i + 1 < len(msg_mjd) else hi
            fresh_end = min(end, msg_mjd[i] + max_age)
            add(
                day_obs,
                start,
                fresh_end,
                exclusive_cause(labels[i]),
                labels[i],
                notes[i],
                "efd",
            )
            add(
                day_obs, max(start, fresh_end), end, "UNKNOWN", "", None, "no_telemetry"
            )

    intervals = (
        pd.DataFrame(rows).sort_values(["day_obs", "start_mjd"]).reset_index(drop=True)
    )
    intervals["measured"] = intervals.provenance == "efd"
    intervals["annotation"] = ""
    # Consecutive FAULT intervals form one physical fault run. Break runs at a
    # night boundary so a fault straddling sunrise is attributed to both nights.
    is_fault = intervals["status_labels"].str.contains("FAULT")
    new_run = (is_fault != is_fault.shift()) | (
        intervals["day_obs"] != intervals["day_obs"].shift()
    )
    intervals["is_fault"] = is_fault
    intervals["fault_run"] = new_run.cumsum().where(is_fault)
    return intervals


def apply_annotations(intervals, annotations, last_dayobs):
    """Relabel intervals inside curated annotation periods.

    Fills unmeasured time and relabels EFD DOWNTIME / WEATHER / UNKNOWN; never
    touches recorded OPERATIONAL, FAULT or IDLE time. Status labels are kept,
    so a real fault under an annotated period still counts as hidden fault.
    """
    intervals = intervals.copy()
    for note in annotations:
        end = note["end"] if note["end"] is not None else last_dayobs
        in_range = intervals.day_obs.between(note["start"], end)
        target = in_range & (
            ~intervals.measured | intervals.cause.isin(ANNOTATION_OVERRIDES)
        )
        intervals.loc[target, "cause"] = note["cause"]
        intervals.loc[target, "provenance"] = "annotation"
        intervals.loc[target, "annotation"] = note["label"]
    return intervals


def night_budget_table(intervals, day_obs_list, night_windows, clamp_now=True):
    """Per-night hours by cause, complete over the whole calendar range.

    Indexed on every night in the range, not just nights that have visits -
    exactly the nights lost to a full-night fault or to weather have none,
    and those are the ones this notebook exists to show.
    """
    budget = intervals.pivot_table(
        index="day_obs", columns="cause", values="hours", aggfunc="sum", fill_value=0.0
    )
    budget = budget.reindex(
        index=day_obs_list, columns=list(BUDGET_CAUSES), fill_value=0.0
    ).fillna(0.0)

    now_mjd = Time.now().tai.mjd
    night_hours, day_obs_mjd = [], []
    for day_obs in budget.index:
        lo, hi = night_windows[day_obs]
        if clamp_now:
            hi = min(hi, now_mjd)
        night_hours.append(max(hi - lo, 0.0) * 24)
        day_obs_mjd.append(dayObsToTime(day_obs).mjd)
    budget["night_hours"] = night_hours
    budget["day_obs_mjd"] = day_obs_mjd
    return budget


if have_status:
    night_windows = build_night_windows(day_obs_nights)
    intervals = build_state_intervals(status_messages, day_obs_nights, night_windows)
    intervals = apply_annotations(intervals, TIME_LOSS_ANNOTATIONS, end_dayobs)
    night_budget = night_budget_table(intervals, day_obs_nights, night_windows)

    # Per-night hours with fresh EFD telemetry, and hours set by an annotation.
    night_budget["efd_hours"] = (
        intervals[intervals.measured]
        .groupby("day_obs")
        .hours.sum()
        .reindex(night_budget.index, fill_value=0.0)
    )
    night_budget["annotation_hours"] = (
        intervals[intervals.provenance == "annotation"]
        .groupby("day_obs")
        .hours.sum()
        .reindex(night_budget.index, fill_value=0.0)
    )
    night_budget["efd_covered"] = (night_budget.night_hours > 0) & (
        night_budget.efd_hours >= MIN_COVERED_FRACTION * night_budget.night_hours
    )
    # Where the hours budget comes from. Annotation wins: on those nights the
    # cause was set by hand even where fresh EFD telemetry also exists.
    night_budget["budget_source"] = np.select(
        [night_budget.annotation_hours > 0, night_budget.efd_covered],
        ["annotation", "efd"],
        default="none",
    )

    # The partition must account for every hour of every night; if it does not,
    # every number below is wrong, so fail loudly rather than plotting it.
    closure = intervals.groupby("day_obs").hours.sum() - night_budget["night_hours"]
    assert np.abs(closure.dropna()).max() < 1e-4, closure
    print(f"partition closes to {np.abs(closure.dropna()).max():.2e} hr on all nights")

    stale = intervals[~intervals.measured]
    print(
        f"\nEFD-covered nights (>= {MIN_COVERED_FRACTION:.0%} fresh telemetry): "
        f"{int(night_budget.efd_covered.sum())} of {len(night_budget)}; "
        f"{stale.hours.sum():.1f} hr of night has no fresh telemetry "
        f"(state older than {MAX_STATE_AGE_HOURS:.0f} hr, or before the topic began)."
    )
    for note in TIME_LOSS_ANNOTATIONS:
        hrs = intervals.loc[intervals.annotation == note["label"], "hours"].sum()
        end = note["end"] or f"end of range ({end_dayobs})"
        print(
            f"  annotation '{note['label']}' {note['start']}-{end}: "
            f"{hrs:.1f} hr relabelled as {note['cause']}"
        )

#### Backfilling nights without EFD telemetry

`lsst.sal.Scheduler.logevent_observatoryStatus` does not extend indefinitely into the past —
it begins **2026-02-25**. Nights before that are not idle nights, they are unmeasured ones,
and leaving them as a solid `UNKNOWN` block would both hide real losses and dilute every
percentage in the range.

Where telemetry is missing, the hours budget is backfilled from the narrative log's
`time_lost` entries (Simonyi only). Two corrections are needed before those numbers can be
used:

- Operators enter `time_lost` free-hand, and the entries **overlap and sometimes exceed the
  night**, for example when one entry covers a multi-day daytime recovery. Each night's
  reported fault+weather is therefore scaled down proportionally to fit within
  `night_hours`, and the clipped excess is reported.
- The cell below calibrates the log against the EFD on nights where *both* exist. In every
  range examined so far the log recovers well under the EFD totals, and weather worst of
  all, because weather closures frequently go unlogged. So the backfilled nights are a
  *lower bound*, and their weather in particular is understated.

The backfilled hours are held in separate `log_fault` / `log_weather` columns and shown in a
distinct hatch, never merged into the EFD-derived causes. Everything requiring per-component
detail — the subsystem breakdown, fault rates, recovery times — is restricted to
EFD-covered nights, because the narrative log's structured `systems` / `subsystems` / `cscs`
fields are rarely filled in; the cell below reports how many entries in the range carry them.

Nights inside a curated annotation are not backfilled: the annotation already says what
the time was.

In [ ]:
def narrative_time_lost(start_dayobs, end_dayobs, narrative_client, night_hours):
    """Per-night fault/weather hours reported by hand in the narrative log.

    Returns a frame indexed like `night_hours` with `log_fault`, `log_weather`,
    and `log_clipped` (hours discarded because the reported total exceeded the
    night). Operators log `time_lost` free-hand: entries overlap and some cover
    daytime recovery, so the raw sum can exceed the night several times over.
    """
    empty = pd.DataFrame(
        0.0,
        index=night_hours.index,
        columns=["log_fault", "log_weather", "log_clipped"],
    )
    logs = narrative_client.query_log(
        rn_dayobs.day_obs_to_time(start_dayobs),
        rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
        {"min_time_lost": "0.00001"},
    )
    if len(logs) == 0:
        return empty
    logs = logs.query("not component.str.contains('AuxTel')")
    if len(logs) == 0:
        return empty

    def to_day_obs(x):
        return rn_dayobs.day_obs_str_to_int(
            rn_dayobs.time_to_day_obs(Time(x.date_begin, format="isot", scale="utc"))
        )

    logs = logs.copy()
    logs["day_obs"] = logs.apply(to_day_obs, axis=1)
    # Data-quality counts, reported by the caller rather than hardcoded.
    n_zero_span = int(
        (
            pd.to_datetime(logs.date_end, utc=True, format="ISO8601")
            <= pd.to_datetime(logs.date_begin, utc=True, format="ISO8601")
        ).sum()
    )
    structured = [c for c in ("systems", "subsystems", "cscs") if c in logs.columns]
    n_structured = int(
        logs[structured]
        .apply(
            lambda col: col.map(
                lambda v: len(v) > 0 if hasattr(v, "__len__") else False
            )
        )
        .any(axis=1)
        .sum()
        if structured
        else 0
    )
    raw = (
        logs.pivot_table(
            index="day_obs",
            columns="time_lost_type",
            values="time_lost",
            aggfunc="sum",
            fill_value=0.0,
        )
        .reindex(night_hours.index, fill_value=0.0)
        .fillna(0.0)
    )
    for column in ("fault", "weather"):
        if column not in raw:
            raw[column] = 0.0
    raw = raw[["fault", "weather"]]

    # Scale each night down so reported loss cannot exceed the night length.
    total = raw.sum(axis=1)
    scale = pd.Series(
        np.where(total > night_hours, night_hours / total.replace(0.0, np.nan), 1.0),
        index=raw.index,
    ).fillna(1.0)
    scaled = raw.mul(scale, axis=0)
    out = pd.DataFrame(
        {
            "log_fault": scaled.fault,
            "log_weather": scaled.weather,
            "log_clipped": (total - night_hours).clip(lower=0.0),
        }
    )
    out.attrs = {
        "n_entries": len(logs),
        "n_zero_span": n_zero_span,
        "n_structured": n_structured,
        "max_ratio": float((total / night_hours.replace(0.0, np.nan)).max()),
    }
    return out


if have_status:
    narrative_lost = narrative_time_lost(
        start_dayobs, end_dayobs, clients["narrative_log"], night_budget.night_hours
    )
    night_budget = night_budget.join(narrative_lost)
    log_stats = narrative_lost.attrs
    if log_stats:
        print(
            f"Narrative log: {log_stats['n_entries']} Simonyi time-loss entries; "
            f"{log_stats['n_zero_span']} have a zero-length date span, "
            f"{log_stats['n_structured']} carry structured systems/subsystems/cscs, "
            f"largest reported loss is {log_stats['max_ratio']:.1f}x its night.\n"
        )

    # Only backfill where there is neither telemetry nor an annotation: on those
    # nights the partition is authoritative and the log is a cross-check.
    uncovered = night_budget.budget_source == "none"
    night_budget["backfilled"] = uncovered
    night_budget["backfill_fault"] = np.where(uncovered, night_budget.log_fault, 0.0)
    night_budget["backfill_weather"] = np.where(
        uncovered, night_budget.log_weather, 0.0
    )
    # Whatever the log does not explain on an uncovered night stays unmeasured.
    night_budget["unmeasured"] = np.where(
        uncovered,
        (
            night_budget.night_hours
            - night_budget.backfill_fault
            - night_budget.backfill_weather
        ).clip(lower=0.0),
        0.0,
    )

    if uncovered.any():
        print(
            f"Backfilled {int(uncovered.sum())} night(s) without telemetry "
            f"({night_budget.loc[uncovered, 'night_hours'].sum():.1f} night hr):"
        )
        print(
            f"  narrative fault   {night_budget.backfill_fault.sum():.1f} hr\n"
            f"  narrative weather {night_budget.backfill_weather.sum():.1f} hr\n"
            f"  still unmeasured  {night_budget.unmeasured.sum():.1f} hr "
            f"({int((night_budget.loc[uncovered, 'log_fault'] + night_budget.loc[uncovered, 'log_weather'] == 0).sum())} "
            "night(s) had no log entry at all)"
        )
        if night_budget.log_clipped.sum() > 0:
            print(
                f"  {night_budget.log_clipped.sum():.1f} hr of reported loss was clipped "
                f"on {int((night_budget.log_clipped > 0).sum())} night(s) for exceeding "
                "the night length"
            )

    # Calibrate the log against the EFD where both exist, so the backfill can be
    # read as the lower bound it is.
    both = night_budget[night_budget.budget_source == "efd"]
    if len(both) > 0 and both.log_fault.sum() > 0:
        print(
            f"\nCalibration on the {len(both)} nights whose budget is EFD-derived "
            "(not annotated) - narrative log recovers:"
        )
        print(
            f"  fault:   {both.log_fault.sum():.1f} / {both.FAULT.sum():.1f} hr "
            f"= {both.log_fault.sum() / max(both.FAULT.sum(), 1e-9):.0%}"
        )
        print(
            f"  weather: {both.log_weather.sum():.1f} / {both.WEATHER.sum():.1f} hr "
            f"= {both.log_weather.sum() / max(both.WEATHER.sum(), 1e-9):.0%}"
        )
        print("=> treat backfilled nights as a lower bound, weather especially.")


def backfilled_intervals(intervals, night_budget, night_windows):
    """`intervals` with each uncovered night's UNKNOWN block replaced by the
    narrative-log backfill, for plotting and gap attribution.

    The partition in `intervals` is left untouched - every fault statistic below
    depends on it and must see only EFD-derived state. This frame is a *display*
    counterpart: it carries the same total hours per night, but on uncovered
    nights those hours are described by the narrative log instead.

    The log cannot say *when* within the night the loss happened - its date_begin
    /date_end spans are unusable (many are zero-length, and some exceed the
    night; the counts are printed above) - so the backfilled blocks are laid out
    in a fixed order from sunset purely to fill the column. Their vertical
    position carries no information, which is why they are drawn hatched.
    """
    uncovered = set(night_budget.index[night_budget.backfilled])
    kept = intervals[~intervals.day_obs.isin(uncovered)].copy()

    rows = []
    for day_obs in sorted(uncovered):
        lo, hi = night_windows[day_obs]
        cursor = lo
        for column, cause in (
            ("unmeasured", "UNKNOWN"),
            ("backfill_weather", "WEATHER"),
            ("backfill_fault", "FAULT"),
        ):
            hours = float(night_budget.loc[day_obs, column])
            if hours <= 0:
                continue
            end = min(cursor + hours / 24.0, hi)
            rows.append(
                {
                    "day_obs": day_obs,
                    "start_mjd": cursor,
                    "end_mjd": end,
                    "hours": (end - cursor) * 24,
                    "cause": cause,
                    "status_labels": "",
                    "note": None,
                    "is_fault": False,
                    "fault_run": np.nan,
                    "measured": False,
                    "annotation": "",
                    "provenance": "narrative" if cause != "UNKNOWN" else "none",
                }
            )
            cursor = end

    combined = pd.concat([kept, pd.DataFrame(rows)], ignore_index=True)
    return combined.sort_values(["day_obs", "start_mjd"]).reset_index(drop=True)


if have_status:
    timeline_intervals = backfilled_intervals(intervals, night_budget, night_windows)
    # Same closure requirement as the partition itself: replacing a night's
    # UNKNOWN block with the backfill must not change its total hours.
    closure_bf = (
        timeline_intervals.groupby("day_obs").hours.sum() - night_budget["night_hours"]
    )
    assert np.abs(closure_bf.dropna()).max() < 1e-4, closure_bf
    n_bf = int((timeline_intervals.provenance == "narrative").sum())
    print(
        f"\ntimeline_intervals: {len(timeline_intervals)} rows, {n_bf} from the "
        "narrative log (night-level, not time-resolved). `intervals` itself is "
        "unchanged, so the fault statistics stay EFD-only."
    )

In [ ]:
if have_status:
    display(
        night_budget[
            list(BUDGET_CAUSES)
            + [
                "night_hours",
                "efd_covered",
                "budget_source",
                "backfill_fault",
                "backfill_weather",
            ]
        ].round(2)
    )

    sources = night_budget.groupby("budget_source").night_hours.agg(["count", "sum"])
    print("Where each night's hours budget comes from:")
    for source, row in sources.iterrows():
        print(
            f"  {source:<10} {int(row['count']):4d} nights  {row['sum']:7.1f} night hr"
        )

    # Quote totals over nights whose budget is measured or curated. Backfilled
    # nights are a lower bound and would understate the loss fractions.
    accounted = night_budget[night_budget.budget_source.isin(["efd", "annotation"])]
    total_hours = accounted.night_hours.sum()
    print(
        f"\nTotals over the {len(accounted)} nights with an EFD or annotated "
        f"budget ({total_hours:.1f} night hr):"
    )
    print(accounted[list(BUDGET_CAUSES)].sum().round(2).to_string())
    lost = accounted[list(LOST_CAUSES)].sum().sum()
    if total_hours > 0:
        print(
            f"\noperational: {accounted.OPERATIONAL.sum():.1f} hr "
            f"({accounted.OPERATIONAL.sum() / total_hours:.0%})  "
            f"lost: {lost:.1f} hr ({lost / total_hours:.0%})  "
            f"unknown: {accounted.UNKNOWN.sum():.1f} hr"
        )
    if night_budget.backfilled.any():
        print(
            "\nThe remaining nights are backfilled from the narrative log and are a "
            "lower bound, so they are left out of these totals."
        )

### Where the night went

Each bar is one night of $-12^\circ$ twilight-to-twilight time, partitioned into the causes
above; the black tick marks the night's full length, so a bar reaching it means every hour is
accounted for. Blue is time the scheduler considered itself operational — everything else is
lost time, and the colour says to what. Hours set by a curated annotation are dotted. The x-axis covers every night in the range, including
nights with no visits at all.

In [ ]:
if have_status:
    # One x position per night in the range, not per night with visits:
    # dropping the visit-less nights would hide the largest losses.
    xticks_budget = night_budget["day_obs_mjd"].values
    xtick_labels_budget = night_budget.index.values

    # A label per night is unreadable much past a fortnight, so thin them to
    # roughly 25 across whatever range was requested. Shared by every plot below.
    tick_step = slice(None, None, max(len(xticks_budget) // 25, 1))
    bar_width = 0.8 if len(xticks_budget) <= 40 else 1.0

    plt.figure(figsize=(16, 8), dpi=200)
    bottom = np.zeros(len(night_budget))
    backfilled = night_budget.backfilled.values
    for cause in BUDGET_CAUSES:
        # On a backfilled night the bars below re-describe the whole night, so
        # drop the partition there or the stack overshoots night_hours.
        hours = np.where(backfilled, 0.0, night_budget[cause].values)
        if hours.sum() <= 0:
            continue
        curated = night_budget.budget_source.values == "annotation"
        plt.bar(
            xticks_budget,
            hours,
            bottom=bottom,
            width=bar_width,
            color=CAUSE_COLORS[cause],
            hatch=[".." if c else None for c in curated],
            edgecolor="0.35",
            linewidth=0.0,
            label=f"{cause} ({hours.sum():.1f} hr)",
        )
        bottom += hours

    # Narrative-log backfill for nights with no telemetry, hatched so it is never
    # mistaken for an EFD-derived measurement.
    for column, color, label in (
        ("backfill_fault", CAUSE_COLORS["FAULT"], "FAULT (narrative log)"),
        ("backfill_weather", CAUSE_COLORS["WEATHER"], "WEATHER (narrative log)"),
        ("unmeasured", "0.85", "no telemetry"),
    ):
        hours = night_budget[column].values
        if hours.sum() <= 0:
            continue
        plt.bar(
            xticks_budget,
            hours,
            bottom=bottom,
            width=bar_width,
            color=color,
            hatch="///",
            edgecolor="0.35",
            linewidth=0.0,
            label=f"{label} ({hours.sum():.1f} hr)",
        )
        bottom += hours

    # The stack must still equal the night, backfill included: an easy mistake
    # here is to leave the whole-night UNKNOWN in place and stack the backfill on
    # top of it, which silently overshoots night_hours.
    assert np.abs(bottom - night_budget["night_hours"].values).max() < 1e-4

    plt.plot(
        xticks_budget,
        night_budget["night_hours"].values,
        "k_",
        ms=18 if len(xticks_budget) <= 40 else 4,
        label="Night length (-12 deg)",
    )
    plt.ylabel("Hours of -12 deg Night")
    plt.xticks(
        xticks_budget[tick_step],
        xtick_labels_budget[tick_step],
        rotation=90.0,
        fontsize=8,
    )
    plt.xlim(xticks_budget[0] - 1.0, xticks_budget[-1] + 1.0)
    plt.legend(loc="upper left", fontsize=7, ncol=4, framealpha=0.9)
    plt.ylim(0.0, night_budget["night_hours"].max() * 1.45)
    plt.minorticks_off()
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig("figures/night_budget_%i-%i.pdf" % (start_dayobs, end_dayobs))
    plt.show()

In [ ]:
def fault_runs_table(intervals):
    """One row per (fault run, CSC), with run hours split equally across CSCs.

    `hours` is the run's full extent; `hours_charged` counts only the part the
    budget charges to FAULT, i.e. excluding time the run spent inside a WEATHER
    or DOWNTIME period.
    """
    records = []
    for run_id, group in intervals.dropna(subset=["fault_run"]).groupby("fault_run"):
        cscs = sorted(set().union(*(set(parse_fault_cscs(n)) for n in group.note)))
        if not cscs:
            # Keep these visible rather than dropping them: a CSC missing from
            # CSC_SUBSYSTEMS would otherwise silently break the totals.
            cscs = [UNATTRIBUTED]
        charged = group.loc[group.cause == "FAULT", "hours"].sum()
        for csc in cscs:
            records.append(
                {
                    "day_obs": int(group.day_obs.iloc[0]),
                    "fault_run": int(run_id),
                    "start_mjd": group.start_mjd.min(),
                    "hours": group.hours.sum() / len(cscs),
                    "hours_charged": charged / len(cscs),
                    "csc": csc,
                    "subsystem": csc_subsystem(csc),
                    "n_cscs": len(cscs),
                    "note": group.note.iloc[0],
                }
            )
    return pd.DataFrame(records)


if have_status:
    fault_runs = fault_runs_table(intervals)

    # Splitting hours across CSCs must conserve them, and the charged hours must
    # be exactly the FAULT column of the budget.
    assert abs(fault_runs.hours_charged.sum() - night_budget.FAULT.sum()) < 1e-6

    fault_by_subsystem = (
        fault_runs.groupby("subsystem").hours_charged.sum().sort_values(ascending=False)
    )
    # Fault that ran under a higher-priority cause, by that cause. Only fresh
    # telemetry can carry a FAULT label, so nothing here is carried across a gap.
    fault_hidden_by_cause = (
        intervals.query("is_fault and cause != 'FAULT'").groupby("cause").hours.sum()
    )
    fault_hidden = fault_hidden_by_cause.sum()

    print("Fault hours charged, by subsystem:")
    print(fault_by_subsystem.round(2).to_string())
    print("\nFault hours charged, by CSC:")
    print(
        fault_runs.groupby("csc")
        .hours_charged.sum()
        .sort_values(ascending=False)
        .round(2)
        .to_string()
    )
    print(
        f"\nA further {fault_hidden:.2f} hr of fault ran while the observatory was "
        "already down for another cause; those hours are charged there, not to "
        "fault, and are not counted above:"
    )
    if len(fault_hidden_by_cause):
        print(fault_hidden_by_cause.round(2).to_string())
    split = fault_runs.query("n_cscs > 1")
    if len(split) > 0:
        print(
            f"{split.fault_run.nunique()} fault run(s) named more than one CSC and had "
            "their hours split equally."
        )

### Fault time by subsystem

Fault hours attributed to the subsystem whose CSC the scheduler recorded as faulted — per
night on the left, totalled over the range on the right. Only the hours the budget charges to
`FAULT` appear here; fault time that ran underneath a weather or downtime period is reported
above instead, so this plot does not blame the system for time already lost to weather. Bars
labelled `unattributed` are fault periods whose note carried no machine-generated CSC clause.

In [ ]:
if have_status:
    subsystem_by_night = (
        fault_runs.pivot_table(
            index="day_obs",
            columns="subsystem",
            values="hours_charged",
            aggfunc="sum",
            fill_value=0.0,
        )
        .reindex(index=night_budget.index, fill_value=0.0)
        .fillna(0.0)
    )
    ordered = fault_by_subsystem[fault_by_subsystem > 0]

    fig, axes = plt.subplots(
        1, 2, figsize=(18, 6), dpi=200, gridspec_kw={"width_ratios": [2.2, 1]}
    )

    ax = axes[0]
    bottom = np.zeros(len(subsystem_by_night))
    for subsystem in ordered.index:
        hours = subsystem_by_night[subsystem].values
        ax.bar(
            xticks_budget,
            hours,
            bottom=bottom,
            width=bar_width,
            color=SUBSYSTEM_COLORS.get(subsystem, "0.6"),
            label=subsystem,
        )
        bottom += hours
    ax.set_ylabel("Fault Hours")
    ax.set_xticks(xticks_budget[tick_step])
    ax.set_xticklabels(xtick_labels_budget[tick_step], rotation=90.0, fontsize=8)
    ax.set_xlim(xticks_budget[0] - 1.0, xticks_budget[-1] + 1.0)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3, axis="y")
    ax.minorticks_off()

    ax = axes[1]
    positions = np.arange(len(ordered))[::-1]
    ax.barh(
        positions,
        ordered.values,
        color=[SUBSYSTEM_COLORS.get(s, "0.6") for s in ordered.index],
    )
    for position, hours in zip(positions, ordered.values):
        ax.text(hours, position, "  %.2f" % hours, va="center", fontsize=9)
    ax.set_yticks(positions)
    ax.set_yticklabels(ordered.index, fontsize=9)
    ax.set_xlabel("Fault Hours (%i-%i)" % (start_dayobs, end_dayobs))
    ax.set_xlim(0.0, ordered.max() * 1.25)
    ax.grid(alpha=0.3, axis="x")
    ax.minorticks_off()

    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig("figures/fault_by_subsystem_%i-%i.pdf" % (start_dayobs, end_dayobs))
    plt.show()

### Fault rate and recovery time per component

The budget above says how many hours each subsystem cost; this says how those hours were
made up — whether a subsystem is failing *often* or failing *slowly*. Two quantities per
component:

- **Fault rate**, faults per 24 hr of *exposure*. Exposure is `OPERATIONAL + FAULT`, i.e.
  time the observatory was actually trying to observe. Hours lost to weather or downtime are
  excluded from the denominator, and — for consistency — fault runs that occurred *entirely*
  within a weather or downtime period are excluded from the numerator too. Counting those
  faults against an exposure that omits their hours would inflate the rate with failures that
  cost no observing time.
- **Recovery time**, the wall-clock duration of each fault run. Note this is the run's *full*
  extent, not the `hours_charged` used in the budget: how long the component took to recover
  is an engineering property of the fault, whereas the charged hours are a time-accounting
  one, and the two differ whenever a fault straddled the start or end of a weather period.

A fault run naming several CSCs contributes one event to *each* of them, and its full
duration to each — unlike the budget, these are not per-CSC shares, so the event counts do
not sum to the number of fault runs and the recovery times must not be added up.

**Read the rates as small-number statistics.** Over a short range, or one with little
observing, most components fault only a handful of times; error bars are $\sqrt{N}$ (Poisson) and the lower arm is clipped at
zero, and a component with `N=1` has no meaningful uncertainty at all. Medians are quoted in
preference to means because the distribution is heavily skewed — a few long faults
dominate any mean.

In [ ]:
def fault_events_table(intervals, exposed_only=True):
    """One row per (fault run, component named in it), for rate/MTTR statistics.

    Unlike `fault_runs_table`, durations here are NOT divided among the CSCs a run
    names: each component gets one event and the run's full duration, since "how
    often did this component fault" and "how long did it take to recover" are per
    component, not shares of a total. Consequently these rows cannot be summed
    into an hours budget - use `fault_runs` for that.
    """
    records = []
    for run_id, group in intervals.dropna(subset=["fault_run"]).groupby("fault_run"):
        charged = group.loc[group.cause == "FAULT", "hours"].sum()
        if exposed_only and charged <= 1e-9:
            # Fault ran entirely inside a WEATHER/DOWNTIME period, whose hours are
            # excluded from the exposure denominator - so exclude it from the
            # numerator as well, or the rate counts failures that cost no time.
            continue
        cscs = sorted(set().union(*(set(parse_fault_cscs(n)) for n in group.note)))
        for csc in cscs or [UNATTRIBUTED]:
            records.append(
                {
                    "day_obs": int(group.day_obs.iloc[0]),
                    "fault_run": int(run_id),
                    "start_mjd": group.start_mjd.min(),
                    "recovery_hours": group.hours.sum(),
                    "charged_hours": charged,
                    "csc": csc,
                    "subsystem": csc_subsystem(csc),
                }
            )
    return pd.DataFrame(records)


def reliability_stats(fault_events, exposure_hours, level="subsystem"):
    """Fault rate per 24 hr of exposure, and recovery-time summary, by `level`."""
    stats = fault_events.groupby(level).agg(
        n_faults=("fault_run", "nunique"),
        median_recovery=("recovery_hours", "median"),
        mean_recovery=("recovery_hours", "mean"),
        max_recovery=("recovery_hours", "max"),
    )
    stats["per_24h"] = stats.n_faults / exposure_hours * 24
    # Poisson sqrt(N); symmetric, so the lower arm is clipped when plotted.
    stats["per_24h_err"] = np.sqrt(stats.n_faults) / exposure_hours * 24
    stats["mtbf_hours"] = exposure_hours / stats.n_faults
    return stats.sort_values("n_faults", ascending=False)


if have_status:
    fault_events = fault_events_table(intervals)

    # Exposure: hours the observatory was actually trying to observe. Weather and
    # downtime hours are excluded, matching the numerator above. Restricted to
    # EFD-covered nights - a night with no telemetry contributes no fault events,
    # so including its hours would silently depress every rate.
    exposure = (night_budget.OPERATIONAL + night_budget.FAULT).where(
        night_budget.efd_covered, 0.0
    )
    exposure_hours = exposure.sum()

    subsystem_reliability = reliability_stats(fault_events, exposure_hours, "subsystem")
    csc_reliability = reliability_stats(fault_events, exposure_hours, "csc")

    n_events = fault_events.fault_run.nunique()
    n_excluded = (
        int(intervals.fault_run.nunique()) - n_events
        if intervals.fault_run.notna().any()
        else 0
    )
    print(
        f"exposure (OPERATIONAL+FAULT): {exposure_hours:.2f} hr over "
        f"{(exposure > 0).sum()} EFD-covered night(s)"
    )
    print(
        f"{n_events} fault runs counted "
        f"({n_excluded} excluded as entirely within weather, downtime or recovery)"
    )
    print(
        f"overall: {n_events / exposure_hours * 24:.2f} faults per 24 hr, "
        f"MTBF {exposure_hours / n_events:.2f} hr, "
        f"median recovery {fault_events.recovery_hours.median() * 60:.1f} min"
    )
    print("\nBy subsystem:")
    display(
        subsystem_reliability[
            [
                "n_faults",
                "per_24h",
                "mtbf_hours",
                "median_recovery",
                "mean_recovery",
                "max_recovery",
            ]
        ].round(3)
    )
    print("By CSC:")
    display(
        csc_reliability[
            ["n_faults", "per_24h", "mtbf_hours", "median_recovery", "mean_recovery"]
        ].round(3)
    )

In [ ]:
if have_status and len(fault_events) > 0:
    ordered_rel = subsystem_reliability.index
    positions = np.arange(len(ordered_rel))[::-1]

    fig, axes = plt.subplots(1, 2, figsize=(17, 6), dpi=200)

    ax = axes[0]
    rate = subsystem_reliability.per_24h.values
    rate_err = subsystem_reliability.per_24h_err.values
    ax.barh(
        positions,
        rate,
        color=[SUBSYSTEM_COLORS.get(s, "0.6") for s in ordered_rel],
    )
    # Clip the lower arm at the bar: sqrt(N) is symmetric and would run negative
    # at low N, which is meaningless for a rate.
    ax.errorbar(
        rate,
        positions,
        xerr=[np.minimum(rate_err, rate), rate_err],
        fmt="none",
        ecolor="0.25",
        elinewidth=1.2,
        capsize=3,
    )
    for position, subsystem in zip(positions, ordered_rel):
        n_faults = subsystem_reliability.loc[subsystem, "n_faults"]
        ax.text(
            subsystem_reliability.loc[subsystem, "per_24h"]
            + subsystem_reliability.loc[subsystem, "per_24h_err"],
            position,
            "  N=%i" % n_faults,
            va="center",
            fontsize=9,
        )
    ax.set_yticks(positions)
    ax.set_yticklabels(ordered_rel, fontsize=9)
    ax.set_xlabel("Faults per 24 hr of Exposure")
    ax.set_xlim(0.0, (rate + rate_err).max() * 1.28)
    ax.grid(alpha=0.3, axis="x")
    ax.minorticks_off()

    ax = axes[1]
    for i, subsystem in enumerate(ordered_rel):
        recovery = np.sort(
            fault_events.loc[
                fault_events.subsystem == subsystem, "recovery_hours"
            ].values
            * 60
        )
        y = np.full(len(recovery), len(ordered_rel) - 1 - i, dtype=float)
        if len(recovery) > 1:
            # Spread ties vertically so overlapping points stay countable.
            y += np.linspace(-0.17, 0.17, len(recovery))
        ax.scatter(
            recovery,
            y,
            s=44,
            color=SUBSYSTEM_COLORS.get(subsystem, "0.6"),
            edgecolor="0.25",
            lw=0.5,
            zorder=3,
        )
        median = np.median(recovery)
        ax.plot(
            [median, median],
            [len(ordered_rel) - 1 - i - 0.33, len(ordered_rel) - 1 - i + 0.33],
            color="black",
            lw=2.4,
            zorder=4,
            label="median" if i == 0 else None,
        )
    ax.set_yticks(positions)
    ax.set_yticklabels(ordered_rel, fontsize=9)
    ax.set_xscale("log")
    ax.set_xlabel("Recovery Time per Fault (minutes)")
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.3, axis="x")

    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(
            "figures/fault_rate_recovery_%i-%i.pdf" % (start_dayobs, end_dayobs)
        )
    plt.show()

### Fault rate and recovery time over the range

Top: every fault as a point at its night, coloured by subsystem, with recovery time on a log
axis — the spread can be several orders of magnitude, from second-long glitches to
full-night faults. The dashed line is the running median over all components, which is
the only trend line the statistics support: per-night counts for any single subsystem are
0–5, so a per-subsystem recovery trend would be noise.

Bottom: cumulative fault rate, i.e. faults so far divided by exposure so far. Reading it
left to right shows whether a component's rate is settling down or climbing as the range
progresses; the early nights are volatile because both numerator and denominator are small.
Components with a single fault are omitted, having no meaningful cumulative rate.

In [ ]:
if have_status and len(fault_events) > 0:
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(16, 10),
        dpi=200,
        sharex=True,
        gridspec_kw={"height_ratios": [1.35, 1]},
    )

    ax = axes[0]
    for subsystem in ordered_rel:
        selection = fault_events[fault_events.subsystem == subsystem].sort_values(
            "day_obs"
        )
        x = np.array([dayObsToTime(d).mjd for d in selection.day_obs], dtype=float)
        # Spread same-night points deterministically, so re-running the notebook
        # reproduces the figure exactly (no RNG).
        for night in np.unique(x):
            same_night = x == night
            if same_night.sum() > 1:
                x[same_night] += np.linspace(-0.3, 0.3, same_night.sum())
        ax.scatter(
            x,
            selection.recovery_hours * 60,
            s=54,
            color=SUBSYSTEM_COLORS.get(subsystem, "0.6"),
            edgecolor="0.25",
            lw=0.5,
            label="%s (N=%i)"
            % (subsystem, subsystem_reliability.loc[subsystem, "n_faults"]),
            zorder=3,
        )
    ordered_events = fault_events.sort_values("start_mjd")
    ax.plot(
        [dayObsToTime(d).mjd for d in ordered_events.day_obs],
        ordered_events.recovery_hours.expanding().median() * 60,
        color="black",
        lw=1.8,
        ls="--",
        zorder=4,
        label="running median (all)",
    )
    ax.set_yscale("log")
    ax.set_ylabel("Recovery Time (min)")
    ax.legend(loc="upper left", fontsize=8, ncol=3)
    ax.grid(alpha=0.3)

    ax = axes[1]
    faults_per_night = fault_events.pivot_table(
        index="day_obs",
        columns="subsystem",
        values="fault_run",
        aggfunc="nunique",
        fill_value=0,
    ).reindex(night_budget.index, fill_value=0)
    cumulative_exposure = exposure.cumsum()
    for subsystem in ordered_rel:
        if subsystem_reliability.loc[subsystem, "n_faults"] < 2:
            continue  # a single fault gives no meaningful cumulative rate
        with np.errstate(divide="ignore", invalid="ignore"):
            cumulative_rate = np.where(
                cumulative_exposure.values > 0,
                faults_per_night[subsystem].cumsum().values
                / cumulative_exposure.values
                * 24,
                np.nan,
            )
        ax.plot(
            xticks_budget,
            cumulative_rate,
            marker="o",
            ms=4,
            color=SUBSYSTEM_COLORS.get(subsystem, "0.6"),
            label=subsystem,
        )
    overall_rate = fault_events.fault_run.nunique() / exposure_hours * 24
    ax.axhline(
        overall_rate,
        color="black",
        ls=":",
        lw=1.5,
        label="all components (%.1f / 24 hr)" % overall_rate,
    )
    ax.set_ylabel("Cumulative Faults / 24 hr")
    ax.set_xticks(xticks_budget[tick_step])
    ax.set_xticklabels(xtick_labels_budget[tick_step], rotation=90.0, fontsize=8)
    ax.set_xlim(xticks_budget[0] - 1.0, xticks_budget[-1] + 1.0)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3)
    ax.minorticks_off()

    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig("figures/fault_rate_trend_%i-%i.pdf" % (start_dayobs, end_dayobs))
    plt.show()

### Rolling fault rate and recovery time

With enough nights the per-fault scatter above becomes hard to read, and a running median
anchored at the start of the range stops responding to change. These panels use a centred
**28-night rolling window** instead, which is long enough that most subsystems accumulate
several faults in it and short enough to follow real change.

Top: rolling fault rate per subsystem — faults in the window divided by exposure in the same
window, so nights lost to weather do not depress the rate. Only subsystems with enough
events over the whole range are drawn. Bottom: rolling median recovery time over all
components, with the interquartile band, on a log axis.

Both are computed over EFD-covered nights only. A window overlapping the pre-telemetry
period is not plotted rather than being scaled up from partial data.

In [ ]:
ROLLING_NIGHTS = 28
MIN_FAULTS_TO_TREND = 5

if have_status and len(fault_events) > 0:
    covered_index = night_budget.index[night_budget.efd_covered]
    # Exposure per night, restricted to covered nights - a window that reaches
    # into the pre-telemetry period would otherwise divide by near-zero.
    exposure_covered = exposure.reindex(covered_index).fillna(0.0)
    faults_covered = (
        fault_events.pivot_table(
            index="day_obs",
            columns="subsystem",
            values="fault_run",
            aggfunc="nunique",
            fill_value=0,
        )
        .reindex(covered_index, fill_value=0)
        .fillna(0)
    )

    window = min(ROLLING_NIGHTS, max(len(covered_index) // 3, 3))
    roll_exposure = exposure_covered.rolling(
        window, center=True, min_periods=window
    ).sum()
    x_covered = np.array([dayObsToTime(d).mjd for d in covered_index])

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(16, 10),
        dpi=200,
        sharex=True,
        gridspec_kw={"height_ratios": [1, 1]},
    )

    ax = axes[0]
    trended = [
        s
        for s in subsystem_reliability.index
        if subsystem_reliability.loc[s, "n_faults"] >= MIN_FAULTS_TO_TREND
        and s in faults_covered.columns
    ]
    for subsystem in trended:
        roll_faults = (
            faults_covered[subsystem]
            .rolling(window, center=True, min_periods=window)
            .sum()
        )
        with np.errstate(divide="ignore", invalid="ignore"):
            roll_rate = np.where(
                roll_exposure > 0, roll_faults / roll_exposure * 24, np.nan
            )
        ax.plot(
            x_covered,
            roll_rate,
            lw=2.0,
            color=SUBSYSTEM_COLORS.get(subsystem, "0.6"),
            label="%s (N=%i)"
            % (subsystem, subsystem_reliability.loc[subsystem, "n_faults"]),
        )
    total_roll = (
        faults_covered.sum(axis=1)
        .rolling(window, center=True, min_periods=window)
        .sum()
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        ax.plot(
            x_covered,
            np.where(roll_exposure > 0, total_roll / roll_exposure * 24, np.nan),
            lw=1.6,
            ls="--",
            color="black",
            label="all components",
        )
    ax.set_ylabel("Faults / 24 hr (%i-night window)" % window)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    ax.grid(alpha=0.3)
    ax.minorticks_off()

    ax = axes[1]
    # Median recovery per night, then a rolling median of the individual faults:
    # reindex events onto nights so the window is in nights, not in events.
    per_night_events = fault_events.groupby("day_obs").recovery_hours.apply(list)
    windowed_median, windowed_lo, windowed_hi = [], [], []
    half = window // 2
    for i, day_obs in enumerate(covered_index):
        lo_i, hi_i = max(i - half, 0), min(i + half + 1, len(covered_index))
        if hi_i - lo_i < window:
            windowed_median.append(np.nan)
            windowed_lo.append(np.nan)
            windowed_hi.append(np.nan)
            continue
        pool = [
            v
            for d in covered_index[lo_i:hi_i]
            if d in per_night_events.index
            for v in per_night_events[d]
        ]
        if len(pool) == 0:
            windowed_median.append(np.nan)
            windowed_lo.append(np.nan)
            windowed_hi.append(np.nan)
        else:
            windowed_median.append(np.median(pool) * 60)
            windowed_lo.append(np.percentile(pool, 25) * 60)
            windowed_hi.append(np.percentile(pool, 75) * 60)
    ax.fill_between(
        x_covered,
        windowed_lo,
        windowed_hi,
        color="tab:blue",
        alpha=0.2,
        label="25th-75th percentile",
    )
    ax.plot(x_covered, windowed_median, lw=2.0, color="tab:blue", label="median")
    ax.set_yscale("log")
    ax.set_ylabel("Recovery Time (min, %i-night window)" % window)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3)

    ax.set_xticks(xticks_budget[tick_step])
    ax.set_xticklabels(xtick_labels_budget[tick_step], rotation=90.0, fontsize=8)
    ax.set_xlim(xticks_budget[0] - 1.0, xticks_budget[-1] + 1.0)

    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig("figures/fault_rate_rolling_%i-%i.pdf" % (start_dayobs, end_dayobs))
    plt.show()

### Visit timeline against observatory state

The visit timeline from `on-sky_utilization.ipynb`, redrawn with the background shaded by the
cause in force at each moment, so a gap can be read off against what the observatory was
doing. Black ticks are survey visits. A column shaded solid with no ticks is a night lost
outright — the case the gap-percentile plot cannot see.

Nights without EFD telemetry are shaded from the narrative-log backfill and **hatched**. The
log records only a per-night total, not when in the night the loss fell — its own
`date_begin`/`date_end` spans are unusable for this, many being zero-length — so on those
columns the *amount* of each colour is meaningful but its vertical position is not. Solid
shading is EFD-derived and time-resolved; dotted shading was set by a curated annotation.

The shading spans $-12^\circ$ twilight only. Its boundaries come from `rubin_nights`
(`Observer.at_site("Rubin")`), while the twilight bands in the utilization notebook's timeline
use `Observer.at_site("LSST")`; the two differ by seconds.

In [ ]:
if have_status:
    plt.figure(figsize=(16, 8), dpi=200)

    # Shade each state interval in place, using the same time convention as the
    # visit timeline in on-sky_utilization.ipynb so the two are comparable. Uses
    # `timeline_intervals`, so nights without telemetry show their narrative-log
    # backfill instead of a featureless UNKNOWN block.
    shade_width = 1.0 if len(xticks_budget) > 40 else 0.9
    for interval in timeline_intervals.itertuples():
        reference = dayObsToTime(interval.day_obs)
        y_start = convertTime(
            Time(interval.start_mjd, format="mjd", scale="tai"), reference
        )
        y_end = convertTime(
            Time(interval.end_mjd, format="mjd", scale="tai"), reference
        )
        backfilled = interval.provenance == "narrative"
        curated = interval.provenance == "annotation"
        plt.bar(
            reference.mjd,
            y_end - y_start,
            bottom=y_start,
            width=shade_width,
            color=CAUSE_COLORS[interval.cause],
            alpha=0.55,
            linewidth=0.0,
            # Hatched where the hours come from the narrative log: the log gives
            # only a night total, so the amount of colour is real but its
            # vertical placement is arbitrary.
            hatch="///" if backfilled else (".." if curated else None),
            edgecolor="0.35" if (backfilled or curated) else None,
        )
    # Legend entries only - the bars above are drawn per interval.
    efd_only = timeline_intervals.query("provenance != 'narrative'")
    for cause in BUDGET_CAUSES:
        if efd_only.query("cause == @cause").hours.sum() > 0:
            plt.bar(np.nan, 0.0, color=CAUSE_COLORS[cause], alpha=0.55, label=cause)
    if (timeline_intervals.provenance == "narrative").any():
        plt.bar(
            np.nan,
            0.0,
            color="0.5",
            alpha=0.55,
            hatch="///",
            edgecolor="0.35",
            label="narrative log (night total only)",
        )
    if (timeline_intervals.provenance == "annotation").any():
        plt.bar(
            np.nan,
            0.0,
            color="0.5",
            alpha=0.55,
            hatch="..",
            edgecolor="0.35",
            label="curated annotation",
        )

    survey_days = np.unique(visits["day_obs"])
    for day_obs in survey_days:
        reference = dayObsToTime(day_obs)
        selection_day_obs = visits["day_obs"] == day_obs
        selection_fbs = np.isin(
            visits[selection_day_obs]["science_program"], SURVEY_PROGRAMS
        )
        y = convertTime(
            Time(visits["exp_midpt_mjd"][selection_day_obs], format="mjd"), reference
        )
        plt.scatter(
            np.tile(reference.mjd, np.sum(selection_fbs)),
            y[selection_fbs],
            marker="_",
            s=20,
            c="black",
            zorder=5,
            label="Survey Programs" if day_obs == survey_days[0] else None,
        )

    plt.legend(loc="upper left", fontsize=8, markerscale=2, ncol=3)
    plt.xlim(xticks_budget[0] - 1.0, xticks_budget[-1] + 1.0)
    plt.ylim(-1.5, 11.0)
    plt.xticks(
        xticks_budget[tick_step],
        xtick_labels_budget[tick_step],
        rotation=90.0,
        fontsize=8,
    )
    plt.ylabel("Time (hrs)")
    plt.minorticks_off()
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig("figures/state_timeline_%i-%i.pdf" % (start_dayobs, end_dayobs))
    plt.show()

### Individual lost-time events

The largest losses, each attributed to a cause. Two kinds of row appear: `visit gap` for a
gap longer than five minutes between consecutive visits, and `no visits` for a night that
produced none at all — those are seeded explicitly, since by construction they generate no
gaps and would otherwise be missing from exactly the nights that matter most.

A long gap can span several state intervals, so `cause` is the state holding the most hours
of the gap rather than whichever happened to be in force at its first instant. `fault_cscs`
lists every CSC recorded as faulted anywhere within the span — populated even when the cause
is weather, since a fault can run underneath it. `note` is the operator/scheduler text, kept
raw so that annotations like intentional engineering faults remain visible.

In [ ]:
def attribute_span(start_mjd, end_mjd, intervals):
    """Dominant cause over [start_mjd, end_mjd], plus CSCs faulted within it."""
    overlap = (
        np.minimum(intervals.end_mjd.values, end_mjd)
        - np.maximum(intervals.start_mjd.values, start_mjd)
    ).clip(min=0)
    if overlap.sum() <= 0:
        # Entirely outside -12 deg night, i.e. observing in twilight, which the
        # partition does not cover. Distinct from the EFD's own UNKNOWN state.
        return pd.Series({"cause": "TWILIGHT", "fault_cscs": "", "note": None})
    covered = intervals[overlap > 0]
    hours = pd.Series(overlap[overlap > 0] * 24, index=covered.index)
    cause = hours.groupby(covered.cause).sum().idxmax()
    cscs = sorted(set().union(*(set(parse_fault_cscs(n)) for n in covered.note), set()))
    dominant = covered[covered.cause == cause]
    note = next((n for n in dominant.note if isinstance(n, str) and n.strip()), None)
    return pd.Series({"cause": cause, "fault_cscs": ",".join(cscs), "note": note})


def lost_time_events(visits, intervals, night_budget, min_gap_s=300.0):
    """Visit gaps over min_gap_s, plus whole nights that produced no visits."""
    rows = []
    for day_obs, group in visits.groupby("day_obs"):
        group = group.sort_values("obs_start_mjd")
        gap_start = group.obs_end_mjd.values[:-1]
        gap_end = group.obs_start_mjd.values[1:]
        gap_seconds = (gap_end - gap_start) * 86400
        for i in np.where(gap_seconds > min_gap_s)[0]:
            rows.append(
                {
                    "day_obs": day_obs,
                    "kind": "visit gap",
                    "start_mjd": gap_start[i],
                    "end_mjd": gap_end[i],
                    "hours": gap_seconds[i] / 3600,
                }
            )
    nights_with_visits = set(visits["day_obs"])
    for day_obs in night_budget.index:
        if day_obs not in nights_with_visits:
            sunset, sunrise = rn_dayobs.day_obs_sunset_sunrise(day_obs, -12)
            rows.append(
                {
                    "day_obs": day_obs,
                    "kind": "no visits",
                    "start_mjd": sunset.tai.mjd,
                    "end_mjd": sunrise.tai.mjd,
                    "hours": night_budget.night_hours[day_obs],
                }
            )
    if len(rows) == 0:
        return pd.DataFrame(
            columns=["day_obs", "kind", "hours", "cause", "fault_cscs", "note"]
        )
    events = pd.DataFrame(rows)
    attribution = events.apply(
        lambda x: attribute_span(x.start_mjd, x.end_mjd, intervals), axis=1
    )
    return events.join(attribution).sort_values(
        "hours", ascending=False, ignore_index=True
    )


if have_status:
    # timeline_intervals, not intervals: on a night without telemetry the loss is
    # described by the narrative-log backfill, so a gap there is attributed to
    # FAULT/WEATHER rather than reported as an uninformative UNKNOWN.
    events = lost_time_events(visits, timeline_intervals, night_budget)
    print(
        f"{len(events)} events totalling {events.hours.sum():.1f} hr "
        f"({len(events.query('kind == \"no visits\"'))} nights with no visits)"
    )
    print("\nHours by attributed cause:")
    print(
        events.groupby("cause")
        .hours.sum()
        .sort_values(ascending=False)
        .round(2)
        .to_string()
    )
    with pd.option_context("display.max_colwidth", 90):
        display(
            events[["day_obs", "kind", "hours", "cause", "fault_cscs", "note"]]
            .head(25)
            .round(2)
        )

### Cross-checks against the other accountings

Two other sources describe the same lost time, and neither agrees with the partition for
reasons that are worth making explicit rather than leaving as a third number that differs.
The `rubin_nights` observatory states are compared column by column — their periods overlap,
so they are an upper bound on each cause and must never be summed — and the narrative log is
what the observing specialists reported by hand.

In [ ]:
if have_status:
    # 1. rubin_nights observatory states. These periods OVERLAP and must not be
    #    summed, so each is compared column by column, not totalled together.
    #    They are upper bounds on the partition: a night-spanning WEATHER period
    #    also covers the OPERATIONAL/FAULT periods inside it.
    state_periods = observatory_status.get_observatory_state_times(
        rn_dayobs.day_obs_to_time(start_dayobs),
        rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
        efd_client,
    )
    if len(state_periods) > 0:
        dome_open = observatory_status.get_dome_open_close(
            rn_dayobs.day_obs_to_time(start_dayobs),
            rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
            efd_client,
        )
        state_periods, state_summary = observatory_status.count_observatory_states(
            state_periods, dome_open
        )
        comparison = pd.DataFrame(
            {
                "weather (partition)": night_budget["WEATHER"],
                "weather_down (states)": state_summary["weather_down"],
                "fault (partition)": night_budget["FAULT"],
                "fault_down (states)": state_summary["fault_down"],
            }
        ).fillna(0.0)
        display(comparison.round(2))
        print(comparison.sum().round(2).to_string())
        diff = (
            comparison["fault_down (states)"].sum()
            - comparison["fault (partition)"].sum()
        )
        under_weather = fault_hidden_by_cause.get("WEATHER", 0.0)
        print(
            f"\nfault_down - fault = {diff:.2f} hr. count_observatory_states "
            "already removes FAULT that overlaps DOWNTIME, so this is expected to "
            f"match the fault the partition charged to WEATHER: {under_weather:.2f} hr."
        )
        print(
            "rubin_nights carries the last state across telemetry gaps, which the "
            f"partition treats as unmeasured after {MAX_STATE_AGE_HOURS:.0f} hr, so "
            "the two can also differ over gaps."
        )

    # 2. The narrative log, i.e. what the observing specialists reported by hand.
    #    Expect this to be much smaller: it is only filled in when someone
    #    writes up a time-loss entry, and its structured subsystem fields are
    #    generally empty, which is why it cannot drive the attribution above.
    time_lost_logs = clients["narrative_log"].query_log(
        rn_dayobs.day_obs_to_time(start_dayobs),
        rn_dayobs.day_obs_to_time(end_dayobs) + TimeDelta(1, format="jd"),
        {"min_time_lost": "0.00001"},
    )
    if len(time_lost_logs) > 0:
        time_lost_logs = time_lost_logs.query("not component.str.contains('AuxTel')")
    if len(time_lost_logs) > 0:
        print("\nNarrative log (Simonyi only), hours reported lost by type:")
        print(
            time_lost_logs.groupby("time_lost_type")
            .time_lost.sum()
            .round(2)
            .to_string()
        )
        print(
            f"({len(time_lost_logs)} entries; compare "
            f"{night_budget.FAULT.sum():.1f} hr fault and "
            f"{night_budget.WEATHER.sum():.1f} hr weather from the EFD partition)"
        )
    else:
        print("\nNo Simonyi narrative-log time-loss entries in this range.")

# Summary

This notebook answers *why* on-sky time went missing, by partitioning every night — including
nights that produced no visits at all, which gap-based views such as `on-sky_utilization.ipynb`
and `efficiency.ipynb` cannot see — into operational time versus time lost to weather,
downtime, or a fault, and attributing the fault hours to a subsystem. Read the accountings
together rather than interchangeably:

- The **partition** (`night_budget`) is exhaustive per night and mutually exclusive by
  construction, so its columns can be summed and compared against `night_hours`. It is the
  right basis for "what fraction of the night did we use".
- The **`rubin_nights` observatory states** cross-check is an upper bound on each cause, since
  those periods overlap; the difference from the partition is the fault time that ran while
  the observatory was already down for weather or downtime.
- The **narrative log** is what observing specialists reported by hand. On nights with
  telemetry it corroborates the totals but cannot drive the attribution; on nights without
  telemetry it is the only source there is, and the backfill it provides is a lower bound —
  the calibration printed above shows how much of the EFD fault and weather time it recovers
  over this range.
- **Curated annotations** label periods no automated source can, such as storm recovery
  and scheduled downtime. They are recorded with their source in `TIME_LOSS_ANNOTATIONS`,
  dotted in the plots, and never overwrite recorded operational, fault or idle time.

The fault-rate and recovery-time views then split those fault hours into *how often* versus
*how slowly* each component failed — the distinction that says whether a subsystem needs
better reliability or faster recovery. A subsystem can dominate the hours budget either way,
and the two point at different work. Over a long range the rolling panels are the ones to
read for trend; the per-fault scatter is for spotting individual events.

Caveats worth restating:

- Telemetry coverage is not the same as the requested range. A state older than
  `MAX_STATE_AGE_HOURS` is treated as unmeasured, not carried forward. Quote loss
  percentages over the nights with an EFD or annotated budget, and treat backfilled
  nights as a floor.
- `FAULT` includes intentional engineering faults, and the attributed subsystem is whichever
  CSC the scheduler recorded as faulted — the component that went to `FAULT`, not necessarily
  the root cause.
- The hours budget and the reliability statistics divide multi-CSC fault runs differently, on
  purpose: `fault_runs` splits a run's hours equally among the CSCs it names, so the totals
  reconcile with `night_budget`, whereas `fault_events` gives each component the run's full
  duration, so counts and recovery times are per component. Neither should be summed across
  the other's grouping.